# STAT 764 · Lab 6 — Fixing the numbers

**Assigned Tue Oct 13 · Due Tue Oct 20, 11:59 pm**

Individual work. Part 1 needs Meeting 11 (Tuesday Oct 13); Parts 2 and 3 need Meeting 12
(Thursday Oct 15), and question **(d)** needs the reading, Van Calster et al. (2019).

Same rules as before: `git pull`, copy into `work/` before editing, Restart & Run All before
submitting, upload the `.ipynb` to Canvas.

Meeting 11 said AUC checks the **order** of the patients and calibration checks the
**numbers**. This lab goes one step further each way: a score that sees both at once and
can be taken apart into the two, a fix for the numbers that leaves the order alone, and the
one rule that decides whether the fix works — **it has to be learned on patients the model
was not fitted on.**

⏱ The whole lab runs in well under a minute.

In [ ]:
import os
import pathlib
import sys

here = pathlib.Path.cwd()
found = ([p for p in [here, *here.parents] if (p / "course" / "stat764.py").exists()]
         + [c.parent.parent for c in here.glob("*/course/stat764.py")])
if os.environ.get("STAT764_REPO"):          # your clone, when it is not above you
    found.insert(0, pathlib.Path(os.environ["STAT764_REPO"]))

if found:                       # you are inside (or just outside) your clone
    sys.path.insert(0, str(found[0] / "course"))
else:                           # Colab, or a copy saved outside the clone
    import urllib.request
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/DataScienceUWL/stat764-fall2026"
        "/main/course/stat764.py", "stat764.py")
    sys.path.insert(0, ".")
    print("  (no local clone found — pulled the helpers from GitHub)")

from stat764 import diabetes

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.optimize import brentq
from scipy.special import expit, logit
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import brier_score_loss, roc_auc_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Meeting 9's problem, exactly: readmitted within 30 days, with the encounters that ended in
# death or hospice removed -- those patients cannot be readmitted.
raw = diabetes()
ENDED = [11, 13, 14, 19, 20, 21]
d = raw[~raw["discharge_disposition_id"].isin(ENDED)].copy()
d["readmit30"] = (d["readmitted"] == "<30").astype(int)

NUM = ["time_in_hospital", "num_lab_procedures", "num_procedures", "num_medications",
       "number_outpatient", "number_emergency", "number_inpatient", "number_diagnoses"]
CAT = ["age", "gender", "race", "admission_type_id", "discharge_disposition_id",
       "admission_source_id", "A1Cresult", "max_glu_serum", "insulin", "change",
       "diabetesMed", "medical_specialty"]
d[CAT] = d[CAT].astype(str)            # the codes are labels, not quantities


def readmission_model(**logistic_args):
    """Meeting 9's pipeline. Anything you pass goes to LogisticRegression -- e.g. C=10_000."""
    return Pipeline([
        ("prep", ColumnTransformer([
            ("num", Pipeline([("fill", SimpleImputer(strategy="median")),
                              ("scale", StandardScaler())]), NUM),
            ("cat", OneHotEncoder(handle_unknown="ignore", min_frequency=20), CAT),
        ])),
        ("model", LogisticRegression(max_iter=2000, **logistic_args)),
    ])


# Meeting 9's split: a quarter of the PATIENTS to the test set, each patient on one side.
tr, te = next(GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=764)
              .split(d, groups=d["patient_nbr"]))
train, test = d.iloc[tr], d.iloc[te]
y_train = train["readmit30"].to_numpy()
y_test = test["readmit30"].to_numpy()
print(f"train {len(train):,} encounters · test {len(test):,} encounters · "
      f"readmitted within 30 days in test: {y_test.mean():.1%}")

Meeting 11's three helpers: `net_per_1000` is Meeting 10's money (dollars saved
minus dollars spent per 1,000 discharges, visiting everyone at or above a threshold);
`by_decile` is the check — ten equal groups, what the model **said** against what **came
back** — now with its grouping split out as `tenths`, so Part 1 can use it; and
`reliability_plot` draws the check.

In [ ]:
READMISSION = 17_700           # dollars -- AHRQ HCUP Statistical Brief #307 (2024), 2020 data
P_STAR = 150 / (0.05 * READMISSION)    # Meeting 10's break-even for a $150 visit: 0.169


def net_per_1000(probs, outcome, threshold=P_STAR, cost=150, prevents=0.05):
    """Meeting 10's money: visit everyone at or above `threshold`. Saved minus spent, per 1,000."""
    visit = probs >= threshold
    saved = (visit & (outcome == 1)).sum() * prevents * READMISSION
    spent = visit.sum() * cost
    return (saved - spent) / len(outcome) * 1000


def tenths(probs):
    """Which tenth (0 to 9) each patient falls in, by probability. A forecast that gives
    everyone the same number cannot be split, so all of its patients land in tenth 0."""
    group = pd.qcut(probs, 10, labels=False, duplicates="drop")
    return np.zeros(len(probs), dtype=int) if np.isnan(group).all() else group.astype(int)


def by_decile(probs, outcome):
    """Ten equal groups by probability: what the model SAID against what HAPPENED in each."""
    group = tenths(probs)
    k = range(group.max() + 1)
    return pd.DataFrame({"said": [probs[group == j].mean() for j in k],
                         "came back": [outcome[group == j].mean() for j in k]},
                        index=pd.RangeIndex(1, group.max() + 2, name="tenth")).round(3)


def reliability_plot(versions, outcome, ax=None, title=""):
    """One point per tenth of patients: x = what the model said, y = what happened.
    A calibrated model's points sit on the dashed diagonal."""
    ax = ax or plt.subplots(figsize=(4.6, 4.6))[1]
    top = 0.0
    for name, probs in versions.items():
        table = by_decile(probs, outcome)
        ax.plot(table["said"], table["came back"], marker="o", label=name)
        top = max(top, table.to_numpy().max())
    top = min(1.0, top * 1.1)
    ax.plot([0, top], [0, top], ls="--", color="gray", label="calibrated")
    ax.set(xlim=(0, top), ylim=(0, top), xlabel="the model said (average in the tenth)",
           ylabel="actually came back (share)", title=title)
    ax.legend(loc="upper left", fontsize=8)
    return ax

Everything below uses that one split and fixed seeds, so your numbers will match mine. Your
job is the experiments, not the plumbing.

---

## Part 1 · The Brier score, taken apart (15 points)

Lab 5 introduced the **Brier score**: the average of (probability − what happened)², where
what happened is 0 or 1. Lower is better. It sees the order *and* the numbers at once — and
with Meeting 11's ten groups it comes apart into three pieces you can read. For tenths
*k* = 1, …, 10, let *w*ₖ be the share of patients in tenth *k*, *s*ₖ what the model **said**
there (its average probability), *c*ₖ the share who **came back**, and *ȳ* the overall rate:

| piece | formula | what it measures |
|---|---|---|
| **reliability** | Σ *w*ₖ (*s*ₖ − *c*ₖ)² | how wrong the **numbers** are: the reliability diagram's distance from its diagonal. 0 is perfect. |
| **resolution** | Σ *w*ₖ (*c*ₖ − *ȳ*)² | how different the tenths turn out to be: the **order**. Bigger is better. |
| **uncertainty** | *ȳ* (1 − *ȳ*) | how hard the outcome is to predict at all. No model changes it. |

**Brier ≈ reliability − resolution + uncertainty** — almost exactly; the small remainder is
the spread of probabilities *inside* each tenth, which ten groups cannot see.

**(i)** Write `brier_parts(probs, outcome)` that returns the three pieces. Group the patients
exactly as `by_decile` does — `tenths(probs)` — and take each tenth's *w*ₖ from its actual
size (a forecast that gives everyone the same number has only one group). Check it on
Meeting 9's model: reliability − resolution + uncertainty should land within 0.001 of
`brier_score_loss`.

**(ii)** Four contestants, all scored on `test`:

| contestant | what it is |
|---|---|
| `model` | `readmission_model()` fitted on `train` — Meeting 9's model |
| `squared` | the model's probabilities, squared |
| `square root` | the square root of the model's probabilities |
| `base rate` | no model: `y_train.mean()` as everyone's probability |

Report **one table**: for each contestant, AUC, the Brier score, the three pieces, and the
**Brier skill score**, 1 − Brier ÷ uncertainty — the share of the base rate's Brier score
the contestant removes (negative means worse than knowing nothing about any patient).

In [ ]:
# YOUR CODE HERE

## Part 2 · Fix the numbers — learned the wrong way, and the right way (20 points)

If the order is good and the numbers are off, you can fix the numbers without touching the
order: learn a map from what the model **said** to what **came back**, on patients whose
outcomes you know, and apply it to new ones. Two maps, both from Meeting 12:

- **Platt scaling** — a logistic regression of the outcome on the model's log-odds. It
  learns a **slope** and an **intercept**; the slope is what Van Calster et al. call the
  *calibration slope*. Slope 1, intercept 0 means nothing needed fixing; a slope below 1
  means the model was **too extreme**, above 1 **too timid**. One straight line on the
  log-odds, so it never changes the order.
- **Isotonic regression** — the best never-decreasing step function from probability to
  rate. Any shape at all, as long as it never goes down; it can tie patients together.

A third, for Part 3: **shift only** — move every log-odds by the same amount so the average
matches what happened. It fixes calibration-in-the-large and nothing else.

In [ ]:
def _log_odds(probs):
    return logit(np.clip(probs, 1e-6, 1 - 1e-6))


def platt(probs_learn, outcome_learn):
    """Platt scaling, learned on (probs_learn, outcome_learn).
    Returns (slope, intercept, fix); fix(probs) applies the map to any probabilities."""
    lr = LogisticRegression(C=1e6, max_iter=1000)          # C=1e6: effectively no penalty
    lr.fit(_log_odds(probs_learn).reshape(-1, 1), outcome_learn)
    def fix(probs):
        return lr.predict_proba(_log_odds(probs).reshape(-1, 1))[:, 1]
    return lr.coef_[0, 0], lr.intercept_[0], fix


def isotonic(probs_learn, outcome_learn):
    """Isotonic regression, learned on (probs_learn, outcome_learn). Returns fix(probs)."""
    iso = IsotonicRegression(out_of_bounds="clip", y_min=0, y_max=1)
    return iso.fit(probs_learn, outcome_learn).predict


def shift_only(probs_learn, outcome_learn):
    """Move every log-odds by one amount so the average probability equals the rate that
    happened. Returns (shift, fix)."""
    z, rate = _log_odds(probs_learn), np.mean(outcome_learn)
    shift = brentq(lambda s: expit(z + s).mean() - rate, -5, 5)
    def fix(probs):
        return expit(_log_odds(probs) + shift)
    return shift, fix


# Part 2's patients: two separate groups of TRAINING patients, each patient in one group only.
patients = np.random.default_rng(6).permutation(train["patient_nbr"].unique())
fit_rows = train[train["patient_nbr"].isin(patients[:1_500])]          # fit the model here
cal_rows = train[train["patient_nbr"].isin(patients[1_500:3_000])]     # learn the fix here
print(f"fit_rows {len(fit_rows):,} discharges · cal_rows {len(cal_rows):,} discharges · "
      f"test {len(test):,}")

Meeting 11's studio built a model that is **too sure of itself**: almost no penalty, very
few patients. Build it again, here on `fit_rows`: `readmission_model(C=10_000)`. Then:

1. Score it on `test`: AUC, Brier, reliability (your `brier_parts`), and its reliability
   diagram. Also its AUC on `fit_rows`, the patients it was fitted on.
2. **The wrong way.** Learn Platt scaling from the model's probabilities for **`fit_rows`** —
   the patients it was fitted on.
3. **The right way.** Learn Platt scaling, and separately isotonic regression, from its
   probabilities for **`cal_rows`** — patients it has never seen.
4. For each fix, report the slope and intercept (Platt only) and, on `test`, AUC, Brier,
   reliability and the average probability. Add one comparison row: Meeting 9's model,
   fitted on all of `train`, from Part 1.
5. One reliability diagram on `test`: the model as fitted, Platt the right way, isotonic the
   right way.

⚠ Learn every fix from the model's probabilities for `fit_rows` or `cal_rows`, then apply it
to the model's probabilities for `test`. Never learn a fix on `test`.

In [ ]:
# YOUR CODE HERE

## Part 3 · The \$150 visit, a year later (15 points)

Meeting 11's studio fitted the model on patients first seen early in the data and scored
patients first seen at the end; the readmission rate had fallen in between. Here are the
same two groups, plus the patients **in between** — the most recent patients you would have
outcomes for when the model goes back into use:

In [ ]:
# Patients are placed by the date of their FIRST stay (encounter_id rises with time), so each
# patient is in one group only.
first_seen = d.groupby("patient_nbr")["encounter_id"].transform("min")
cut_60, cut_75 = np.quantile(d["encounter_id"], [0.60, 0.75])
early = d[first_seen <= cut_60]                              # fit the model here
recent = d[(first_seen > cut_60) & (first_seen <= cut_75)]   # the newest outcomes you have
later = d[first_seen > cut_75]                               # the patients you will score
for name, group in (("early", early), ("recent", recent), ("later", later)):
    print(f"  {name:7} {len(group):>7,} stays · readmitted {group['readmit30'].mean():.1%}")

Fit `readmission_model()` on `early` and score `later`. Then apply Meeting 10's \$150 rule —
visit everyone at or above `P_STAR` — to `later`, with the probabilities:

1. as fitted;
2. after Platt scaling learned on `early` — the patients the model was fitted on;
3. after Platt scaling learned on `recent`;
4. after `shift_only` learned on `recent`;
5. **in hindsight** — Platt scaling learned on `later` itself;
6. **in hindsight** — as fitted, but at the threshold in `GRID` that nets the most on
   `later`, found by searching `later`.

Report **one table**: for each row, AUC, the average probability (against the share who came
back), the share visited, and the net per 1,000 discharges. Also the slope and intercept of
Platt scaling learned on `early`, on `recent` and on `later`, and the shift learned on `recent`.

In [ ]:
GRID = np.round(np.arange(0.05, 0.60, 0.005), 3)      # thresholds for row 6's search

In [ ]:
# YOUR CODE HERE

## Part 4 · Written (25 points)

Three or four sentences each, and **every answer except (d) points at a number in your own
output.** Under each question, *A full answer* says what earns full credit — the same lines
are on the Canvas rubric.

**(a)** From Part 1. Which of the three pieces is the same for `model`, `squared` and
`square root`, and why must it be? Which piece makes `squared` and `square root` lose to the
base rate — a forecast that knows nothing about any patient — when their AUC is 0.655? And
what does the model's Brier skill score say about how much it knows?

*A full answer:* names resolution as the shared piece and explains it through the order (same
order, same tenths, same *came back*); names reliability as the piece that grows, with the
numbers; and quotes the model's skill score and says what it means in words.

**(b)** From Part 2. Why does Platt scaling learned on `fit_rows` barely change anything,
while the same method learned on `cal_rows` does? What does the right-way slope say about the
model? On your diagram, where does Platt's line still miss, and why can isotonic regression
follow the curve there when Platt cannot? Did either fix raise the AUC — and what would?

*A full answer:* uses the AUC on `fit_rows` against the AUC on `test` to say the model looks
better on its own patients, so the fix learned there finds nothing to fix; quotes the two
slopes and reads the right-way slope (below 1: too extreme); names where Platt still misses
(the top tenth) and says one straight line on the log-odds cannot bend, while isotonic can
take any never-decreasing shape; and says the fixes change the numbers, not the order, so AUC
stays — more patients (the comparison row) is what raises it.

**(c)** From Part 3. What is the fix learned on `recent` worth per 1,000 discharges, against
doing nothing and against the two hindsight rows? Why is the fix learned on `early` worth
nothing? Why does the probability still run high after the `recent` fix?

*A full answer:* quotes the nets of rows 1, 3, 5 and 6; says the model is already calibrated
on the patients it was fitted on, so the `early` fix learns slope ≈ 1 and intercept ≈ 0; and
explains the leftover overestimate through the rate still falling between `recent` and
`later`. Notes the hindsight rows cannot be had in advance.

**(d)** 📗 **From the reading.** Van Calster et al. (2019) describe four levels of
calibration. Name them, say what each requires, and say which level Meeting 11's "average
said against average came back" checks and which a reliability diagram checks. Why do they
call the strongest level utopic? Then their cardiovascular example, QRISK2 against the NICE
Framingham score: what were the two AUCs, how many men per 1,000 did each select at a 20%
risk threshold, and what does that say about choosing a model by AUC?

*A full answer:* mean, weak, moderate and strong calibration, each defined; the average check
is mean calibration (calibration-in-the-large) and the diagram is moderate; strong needs every
combination of predictor values to be right; 0.771 against 0.776 and 110 against 206 per
1,000 — the overestimating score, with the higher AUC, would treat almost twice as many. No
number from your own output needed.

⚠ Answer (d) from the reading. It is there, and it is short.

**(e)** Your hospital is offered a readmission model built at another hospital. The vendor
writes: *"AUC 0.66 on 25,000 patients — you can use our probabilities as they are."* Write
the three sentences you would send back, each naming a number from this lab.

*A full answer:* three sentences, each naming a number from this lab, that say what AUC
cannot tell the hospital, what it should check on its own patients, and how it would fix the
numbers if they are off.

### Your answers

*(double-click to edit)*

**(a)**

**(b)**

**(c)**

**(d)**

**(e)**

---

## Before you submit

- [ ] **Run → Restart Kernel and Run All Cells**
- [ ] Part 1 has `brier_parts` checked against `brier_score_loss`, and the four-contestant
      table with the skill score
- [ ] Part 2 has the table of fixes (with Platt's slopes and intercepts), the comparison
      row, and the reliability diagram
- [ ] Part 3 has the six-row table and the learned slopes, intercepts and shift
- [ ] All five written answers are filled in, each pointing at a number
- [ ] Uploaded the `.ipynb` to Canvas

**75 points.**